# 03 — Modeling: Baseline, XGBoost, Threshold & Cost Analysis, SHAP

Uses `src/model.py` and `src/scoring.py` directly rather than duplicating logic — this notebook
is the narrative walkthrough, the `.py` modules are the reusable pipeline (also what the
Streamlit dashboard imports).

In [ ]:
import sys
sys.path.insert(0, "../src")
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

import model as m
from features import get_model_feature_columns, INTERPRETABLE_MODEL_FEATURES

df = pd.read_parquet("../data/processed/transactions_features.parquet")
train, test = m.chronological_split(df)

## Why a chronological split?

A fraud model is deployed to score **future** transactions. A random split lets the model
see transactions from the same time window as its test set, which inflates apparent
performance relative to real deployment. Splitting on `TransactionDT` (first 80% -> train,
last 20% -> test) evaluates the model the way it will actually be used.

## Baseline: Logistic Regression (interpretable features only)

In [ ]:
logreg_pipe, logreg_proba, logreg_metrics = m.train_baseline_logreg(train, test)
logreg_metrics

## Main model: XGBoost (interpretable + masked features)

In [ ]:
feature_cols = get_model_feature_columns(df, include_masked=True)
xgb_model, xgb_proba, xgb_metrics = m.train_xgboost(train, test, feature_cols)
print(f"Logistic Regression -> ROC-AUC {logreg_metrics['roc_auc']:.3f}, PR-AUC {logreg_metrics['pr_auc']:.3f}")
print(f"XGBoost             -> ROC-AUC {xgb_metrics['roc_auc']:.3f}, PR-AUC {xgb_metrics['pr_auc']:.3f}")

PR-AUC (precision-recall) matters far more than accuracy or even ROC-AUC here: fraud is ~3.5% of transactions, so a model predicting "never fraud" would score 96.5% accuracy while catching zero fraud.

## Threshold sweep

Default 0.5 is not necessarily the right operating point. Sweeping thresholds shows the alert-volume / precision / recall trade-off directly.

In [ ]:
y_test = test["isFraud"].values
sweep = m.threshold_sweep(y_test, xgb_proba)
sweep.round(4)

In [ ]:
fig, ax = plt.subplots()
ax.plot(sweep["threshold"], sweep["precision"], marker="o", label="Precision")
ax.plot(sweep["threshold"], sweep["recall"], marker="o", label="Recall")
ax.set_xlabel("Threshold"); ax.set_ylabel("Score"); ax.legend(); ax.set_title("Precision / Recall vs. Threshold")
plt.show()

## Fixed investigation capacity: Fraud Capture@500/day

Assume investigators can only review 500 alerts per day. How much fraud gets caught?

In [ ]:
cap500 = m.capacity_at_k(y_test, xgb_proba, k=500)
print("Single global top-500 cut:", cap500)

daily = m.daily_capacity_simulation(test, xgb_proba, capacity=500)
print(f"\nAveraged daily recall at 500/day capacity: {daily['recall'].mean():.1%}")
daily.head()

## Cost-sensitive threshold optimization

**Scenario assumptions, not real industry costs:** false negative = \$500, false positive review = \$10.

In [ ]:
cost_curve = m.cost_sensitive_threshold(y_test, xgb_proba)
best = cost_curve.loc[cost_curve["total_cost"].idxmin()]
print(best)

fig, ax = plt.subplots()
ax.plot(cost_curve["threshold"], cost_curve["total_cost"])
ax.axvline(best["threshold"], color="green", linestyle="--", label=f"cost-optimal = {best['threshold']:.2f}")
ax.set_xlabel("Threshold"); ax.set_ylabel("Estimated scenario cost ($)"); ax.legend()
ax.set_title("Cost-sensitive threshold optimization")
plt.show()

## SHAP: global feature importance

Computed on a 20k sample of the test set for speed. Masked (`C/D/M/V/id_`) features are labeled generically — no invented per-column meaning.

In [ ]:
explainer, shap_values, X_sample = m.compute_shap(xgb_model, test[feature_cols], sample_size=20000)
importance = m.global_feature_importance(shap_values, feature_cols)
importance.head(20)

In [ ]:
shap.summary_plot(shap_values, X_sample, max_display=20, show=False)
plt.tight_layout()
plt.savefig("../outputs/shap_summary.png", dpi=120, bbox_inches="tight")
plt.show()

## Local explanation: one flagged transaction

In [ ]:
top_idx = np.argsort(-xgb_proba)[0]
row_id = test.iloc[top_idx]["TransactionID"]
print(f"TransactionID {row_id}, fraud_probability={xgb_proba[top_idx]:.4f}, isFraud={test.iloc[top_idx]['isFraud']}")

x_row = test.iloc[[top_idx]][feature_cols]
sv_row = explainer(x_row)
shap.plots.waterfall(sv_row[0], max_display=12, show=True)

## Summary

This notebook produces the same artifacts as `python src/model.py` + `python src/scoring.py`
(model bundle, threshold sweep, cost curve, feature importance) — see `outputs/model_summary.json`
for the run this project's README quotes.